# Unify NYC TLC Yellow Taxi

Transform Data (Parquet → Curated Delta)

## Purpose
This notebook transforms the raw NYC TLC Yellow Taxi **monthly parquet files** (2009–2025) into a **single, curated Delta table** with a stable, analytics-ready schema.

The source data spans more than 15 years and includes:
- Multiple physical schemas
- Evolving column names
- Changing data types
- New fields introduced over time
- Legacy GPS-based records (pre–LocationID era)

This notebook handles all of that explicitly and deterministically.

---

## What this notebook does

1. **Reads all raw parquet files** from the Lakehouse Files area  
   (`Files/TLC_Trip_Record_Data/year=YYYY/month=MM/`)

2. **Standardizes column names and data types**
   - Handles historical naming differences
   - Casts all fields into a canonical schema
   - Preserves legacy GPS columns for early years

3. **Safely unifies schema variants**
   - Groups files by *physical parquet schema*
   - Processes each group in batches for performance
   - Falls back gracefully if a group contains edge cases

4. **Creates a curated Delta table**
   - One table spanning the full history
   - Partitioned by `year` and `month`
   - Suitable as a Silver layer or direct foundation for a Gold model

---

## Output

**Delta table**
- `tlc_yellow_tripdata_unified`

**Characteristics**
- Single, stable schema across all years
- No silent data loss
- Preserves historical truth
- Optimized for downstream analytics and semantic modeling

---

## Design principles

- **Correctness over convenience**  
  No assumptions are made about historical schemas.

- **Idempotent and repeatable**  
  Safe to re-run as new months are added.

- **Explicit schema evolution**  
  All field mappings are defined in code.

- **Analytics-first**  
  The output is designed for Power BI / Fabric semantic models.

---

## Prerequisites

- A Fabric Spark notebook attached to the Lakehouse containing:
  `Files/TLC_Trip_Record_Data/`
- All required parquet files already downloaded
- Sufficient Spark capacity to process the full dataset

---

## Notes

- This notebook intentionally does **not** create dimension tables.
- Mapping tables (Vendor, Rate Code, Payment Type, Taxi Zone, Date)
  are created in a separate notebook to keep responsibilities clear.
- A Gold fact table can be derived from the curated Delta table
  once all dimensions are in place.

---


In [ ]:
from pyspark.sql import functions as F, types as T
import re
from typing import List, Dict, Tuple

## Configuration

Set the input folder and output Delta table name. This notebook expects the raw monthly parquet files to already exist in:

`Files/TLC_Trip_Record_Data/year=YYYY/month=MM/`

In [ ]:
# ----------------------------
# Configuration
# ----------------------------
SOURCE_ROOT = "Files/TLC_Trip_Record_Data"     # Lakehouse Files path (Spark)
TARGET_TABLE = "tlc_yellow_tripdata_unified2"  # Delta table name (managed)

# Batch strategy
PROCESS_LARGEST_GROUPS_FIRST = True
FALLBACK_CHUNK_SIZE = 10     # if a group fails, retry in chunks of N files, then single-file fallback

# Spark settings (avoid schema merge across parquet files)
spark.conf.set("spark.sql.parquet.mergeSchema", "false")

## Discover files and schema variants

We list all parquet files recursively and group them by *physical parquet schema* (column names + physical types).  
This prevents Spark from failing when it tries to merge incompatible parquet column types across many files.

In [ ]:
# ----------------------------
# File listing (recursive) using Hadoop FS (no dbutils required)
# ----------------------------
jvm = spark._jvm
hconf = spark._jsc.hadoopConfiguration()
fs = jvm.org.apache.hadoop.fs.FileSystem.get(hconf)
Path = jvm.org.apache.hadoop.fs.Path

def list_parquet_files(root_path: str) -> List[str]:
    out = []
    stack = [Path(root_path)]
    while stack:
        p = stack.pop()
        for status in fs.listStatus(p):
            if status.isDirectory():
                stack.append(status.getPath())
            else:
                s = status.getPath().toString()
                if s.lower().endswith(".parquet"):
                    out.append(s)
    return out

paths = list_parquet_files(SOURCE_ROOT)

ym = re.compile(r"/year=(\d{4})/month=(\d{2})/")
def ym_key(p: str) -> Tuple[int, int]:
    m = ym.search(p)
    return (int(m.group(1)), int(m.group(2))) if m else (9999, 99)

paths = sorted(paths, key=ym_key)
print("Parquet files found:", len(paths))
print("First:", paths[0] if paths else "NONE")
print("Last: ", paths[-1] if paths else "NONE")

In [ ]:
# ----------------------------
# Group by *physical* parquet schema to avoid cross-file type conversion errors
# Signature: sorted list of "col:type" from the parquet footer schema
# ----------------------------
def physical_sig(path: str) -> str:
    sch = spark.read.parquet(path).schema
    items = sorted([f"{f.name}:{f.dataType.simpleString()}" for f in sch.fields])
    return "|".join(items)

groups: Dict[str, List[str]] = {}
for p in paths:
    sig = physical_sig(p)
    groups.setdefault(sig, []).append(p)

group_items = list(groups.items())
if PROCESS_LARGEST_GROUPS_FIRST:
    group_items.sort(key=lambda kv: -len(kv[1]))

print("Distinct physical schema groups:", len(group_items))
print("Top 10 group sizes:", [len(v) for _, v in group_items[:10]])

## Standardize to a canonical schema

The `standardize()` function maps all known historical variants to one stable schema:
- Modern (LocationID + tpep_* fields)
- Legacy (GPS coordinates + Trip_* datetime fields)
- Naming differences (Airport_fee vs airport_fee, surcharge vs extra, etc.)
- Introduced-later fields are set to NULL before they existed

In [ ]:
# ----------------------------
# Standardization logic
# - Works across all discovered schema variants
# - Never references a column unless it exists (prevents AnalysisException)
# ----------------------------
from pyspark.sql import DataFrame

CANONICAL_COLS = [
    # modern dictionary-aligned
    "VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime",
    "passenger_count", "trip_distance", "RatecodeID", "store_and_fwd_flag",
    "PULocationID", "DOLocationID", "payment_type",
    "fare_amount", "extra", "mta_tax", "tip_amount", "tolls_amount",
    "improvement_surcharge", "congestion_surcharge", "airport_fee", "cbd_congestion_fee",
    "total_amount",
    # lineage + legacy
    "vendor_name_raw",
    "pickup_datetime_legacy", "dropoff_datetime_legacy",
    "pickup_longitude", "pickup_latitude", "dropoff_longitude", "dropoff_latitude",
    # partitions
    "year", "month"
]

def standardize(df_in: DataFrame) -> DataFrame:
    df = (
        df_in.withColumn("_path", F.input_file_name())
             .withColumn("year", F.regexp_extract(F.col("_path"), r"/year=(\d{4})/", 1).cast("int"))
             .withColumn("month", F.regexp_extract(F.col("_path"), r"/month=(\d{2})/", 1).cast("int"))
             .drop("__index_level_0__")
    )

    cols = set(df.columns)

    def c(name: str):
        return F.col(name) if name in cols else F.lit(None)

    def co(*names):
        use = [F.col(n) for n in names if n in cols]
        return F.coalesce(*use) if use else F.lit(None)

    # Vendor normalization (supports modern VendorID, legacy vendor_id, or legacy vendor_name)
    vendor_name_in = c("vendor_name").cast("string")
    vendor_id_in = c("vendor_id").cast("int")
    vendorid_in = c("VendorID").cast("int")

    vendorid = (
        F.when(vendorid_in.isNotNull(), vendorid_in)
         .when(vendor_id_in.isNotNull(), vendor_id_in)
         .when(vendor_name_in.isin("CMT"), F.lit(1))
         .when(vendor_name_in.isin("VTS"), F.lit(2))
         .otherwise(F.lit(None).cast("int"))
    )

    out = (
        df
        .withColumn("VendorID", vendorid)
        .withColumn("vendor_name_raw", vendor_name_in)

        # Modern columns (nullable; cast to canonical types)
        .withColumn("tpep_pickup_datetime", c("tpep_pickup_datetime").cast("timestamp"))
        .withColumn("tpep_dropoff_datetime", c("tpep_dropoff_datetime").cast("timestamp"))
        .withColumn("PULocationID", c("PULocationID").cast("int"))
        .withColumn("DOLocationID", c("DOLocationID").cast("int"))
        .withColumn("RatecodeID", c("RatecodeID").cast("int"))
        .withColumn("store_and_fwd_flag", c("store_and_fwd_flag").cast("string"))
        .withColumn("payment_type", c("payment_type").cast("int"))
        .withColumn("passenger_count", c("passenger_count").cast("int"))
        .withColumn("trip_distance", c("trip_distance").cast("double"))

        .withColumn("fare_amount", c("fare_amount").cast("double"))
        .withColumn("extra", c("extra").cast("double"))
        .withColumn("mta_tax", c("mta_tax").cast("double"))
        .withColumn("tip_amount", c("tip_amount").cast("double"))
        .withColumn("tolls_amount", c("tolls_amount").cast("double"))
        .withColumn("improvement_surcharge", c("improvement_surcharge").cast("double"))
        .withColumn("total_amount", c("total_amount").cast("double"))
        .withColumn("congestion_surcharge", c("congestion_surcharge").cast("double"))
        .withColumn("airport_fee", co("airport_fee", "Airport_fee").cast("double"))
        .withColumn("cbd_congestion_fee", c("cbd_congestion_fee").cast("double"))

        # Legacy GPS-era and legacy datetime columns
        .withColumn("pickup_datetime_legacy", co("pickup_datetime", "Trip_Pickup_DateTime").cast("timestamp"))
        .withColumn("dropoff_datetime_legacy", co("dropoff_datetime", "Trip_Dropoff_DateTime").cast("timestamp"))
        .withColumn("pickup_longitude", co("pickup_longitude", "Start_Lon").cast("double"))
        .withColumn("pickup_latitude",  co("pickup_latitude",  "Start_Lat").cast("double"))
        .withColumn("dropoff_longitude",co("dropoff_longitude","End_Lon").cast("double"))
        .withColumn("dropoff_latitude", co("dropoff_latitude", "End_Lat").cast("double"))

        # Legacy numeric fields -> modern equivalents where modern is NULL
        .withColumn("fare_amount", F.coalesce(F.col("fare_amount"), c("Fare_Amt").cast("double")))
        .withColumn("tip_amount", F.coalesce(F.col("tip_amount"), c("Tip_Amt").cast("double")))
        .withColumn("tolls_amount", F.coalesce(F.col("tolls_amount"), c("Tolls_Amt").cast("double")))
        .withColumn("total_amount", F.coalesce(F.col("total_amount"), c("Total_Amt").cast("double")))
        .withColumn("trip_distance", F.coalesce(F.col("trip_distance"), c("Trip_Distance").cast("double")))
        .withColumn("payment_type", F.coalesce(F.col("payment_type"), c("Payment_Type").cast("int")))
        .withColumn("passenger_count", F.coalesce(F.col("passenger_count"), c("Passenger_Count").cast("int")))
        .withColumn("RatecodeID", F.coalesce(F.col("RatecodeID"), co("rate_code", "Rate_Code").cast("int")))
        .withColumn("store_and_fwd_flag", F.coalesce(F.col("store_and_fwd_flag"), c("store_and_forward").cast("string")))
        .withColumn("extra", F.coalesce(F.col("extra"), c("surcharge").cast("double")))
    )

    # Enforce known "introduced later" fields
    out = (
        out
        .withColumn("improvement_surcharge",
                    F.when(F.col("year") < 2015, F.lit(None).cast("double")).otherwise(F.col("improvement_surcharge")))
        .withColumn("cbd_congestion_fee",
                    F.when(F.col("year") < 2025, F.lit(None).cast("double")).otherwise(F.col("cbd_congestion_fee")))
    )

    return out.select(*CANONICAL_COLS)

In [ ]:
# ----------------------------
# Batch processing helpers with fallback
# ----------------------------
def write_batch(batch_paths: List[str], mode: str):
    df_batch = spark.read.format("parquet").load(batch_paths)
    df_std = standardize(df_batch)
    (df_std.write.format("delta")
          .mode(mode)
          .option("overwriteSchema", "true" if mode == "overwrite" else "false")
          .partitionBy("year", "month")
          .saveAsTable(TARGET_TABLE))

failed_files: List[Tuple[str, str]] = []
writes_done = 0

def safe_write(paths_to_write: List[str], mode: str):
    global writes_done
    try:
        write_batch(paths_to_write, mode)
        writes_done += 1
        return True
    except Exception as e:
        return False, repr(e)

## Write curated Delta table

We process schema groups in batches for speed.  
If a group fails, we automatically fall back to smaller chunks and then single-file processing for edge cases.

Output:
- `tlc_yellow_tripdata_unified` (partitioned by `year`, `month`)

In [ ]:
# ----------------------------
# Execute transform → curated Delta table
# ----------------------------
total_files = sum(len(v) for _, v in group_items)
print(f"Processing {total_files} files across {len(group_items)} physical schema groups...")

table_written = False

for gi, (sig, gpaths) in enumerate(group_items):
    # Determine write mode
    mode = "overwrite" if not table_written else "append"

    ok = safe_write(gpaths, mode)
    if ok is True:
        table_written = True
        print(f"Group {gi+1}/{len(group_items)} OK | files={len(gpaths)} | mode={mode}")
        continue

    # Group failed → fallback chunking
    _, err = ok
    print(f"Group {gi+1}/{len(group_items)} FAILED | files={len(gpaths)} | err={err[:200]}...")
    chunk = FALLBACK_CHUNK_SIZE

    for start in range(0, len(gpaths), chunk):
        sub = gpaths[start:start+chunk]
        mode2 = "overwrite" if not table_written else "append"
        ok2 = safe_write(sub, mode2)
        if ok2 is True:
            table_written = True
            continue

        # Chunk failed → single-file fallback
        _, err2 = ok2
        for p in sub:
            mode3 = "overwrite" if not table_written else "append"
            ok3 = safe_write([p], mode3)
            if ok3 is True:
                table_written = True
            else:
                _, err3 = ok3
                failed_files.append((p, err3))
                print("FAILED FILE:", p, err3[:200])

print("Transform complete.")
print("Failed files:", len(failed_files))
if failed_files:
    for p, e in failed_files[:20]:
        print(" ", p, "->", e[:200])

## Validate

We verify:
- Table schema
- Row counts per month (partitions)

In [ ]:
# ----------------------------
# Quick validation: partitions and schema
# ----------------------------
t = spark.table(TARGET_TABLE)
t.printSchema()

display(
    t.groupBy("year", "month")
     .count()
     .orderBy("year", "month")
)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

# Spark-readable path (NOT /lakehouse/default/Files/...)
BASE_PATH = "Files/TLC_Trip_Record_Data"
DELTA_TABLE = "tlc_yellow_tripdata"

df_raw = spark.read.format("parquet").load(BASE_PATH)

target_cols = [
    ("VendorID", T.IntegerType()),
    ("tpep_pickup_datetime", T.TimestampType()),
    ("tpep_dropoff_datetime", T.TimestampType()),
    ("passenger_count", T.IntegerType()),
    ("trip_distance", T.DoubleType()),
    ("RatecodeID", T.IntegerType()),
    ("store_and_fwd_flag", T.StringType()),
    ("PULocationID", T.IntegerType()),
    ("DOLocationID", T.IntegerType()),
    ("payment_type", T.IntegerType()),
    ("fare_amount", T.DoubleType()),
    ("extra", T.DoubleType()),
    ("mta_tax", T.DoubleType()),
    ("tip_amount", T.DoubleType()),
    ("tolls_amount", T.DoubleType()),
    ("improvement_surcharge", T.DoubleType()),
    ("total_amount", T.DoubleType()),
    ("congestion_surcharge", T.DoubleType()),
    ("airport_fee", T.DoubleType()),
    ("cbd_congestion_fee", T.DoubleType()),
]

partition_cols = [
    ("year", T.IntegerType()),
    ("month", T.IntegerType()),
]

def ensure_and_cast(df, col_name, data_type):
    if col_name not in df.columns:
        return df.withColumn(col_name, F.lit(None).cast(data_type))
    return df.withColumn(col_name, F.col(col_name).cast(data_type))

df_std = df_raw
for name, dtype in target_cols + partition_cols:
    df_std = ensure_and_cast(df_std, name, dtype)

df_std = (
    df_std
    .withColumn(
        "improvement_surcharge",
        F.when(F.col("year") < 2015, F.lit(None).cast("double")).otherwise(F.col("improvement_surcharge"))
    )
    .withColumn(
        "cbd_congestion_fee",
        F.when(F.col("year") < 2025, F.lit(None).cast("double")).otherwise(F.col("cbd_congestion_fee"))
    )
)

select_order = [c for c, _ in target_cols] + [c for c, _ in partition_cols]
df_final = df_std.select(*select_order)

(
    df_final.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .partitionBy("year", "month")
            .saveAsTable(DELTA_TABLE)
)

print(f"Created Delta table: {DELTA_TABLE}")

In [ ]:
DELTA_TABLE = "tlc_yellow_tripdata"

# Schema check
spark.table(DELTA_TABLE).printSchema()

# Row counts by month (spot-check)
display(
    spark.table(DELTA_TABLE)
         .groupBy("year", "month")
         .count()
         .orderBy("year", "month")
)

In [ ]:
import re
from pyspark.sql import types as T

BASE = "Files/TLC_Trip_Record_Data"

# Use Hadoop FileSystem to list recursively
jvm = spark._jvm
hconf = spark._jsc.hadoopConfiguration()
fs = jvm.org.apache.hadoop.fs.FileSystem.get(hconf)
Path = jvm.org.apache.hadoop.fs.Path

def list_parquet_files(root_path: str):
    out = []
    stack = [Path(root_path)]
    while stack:
        p = stack.pop()
        for status in fs.listStatus(p):
            if status.isDirectory():
                stack.append(status.getPath())
            else:
                path_str = status.getPath().toString()
                if path_str.lower().endswith(".parquet"):
                    out.append(path_str)
    return out

all_paths = list_parquet_files(BASE)
print("Parquet files found:", len(all_paths))
print("Example path:", all_paths[0] if all_paths else "NONE")

# Extract year/month from hive folders
ym = re.compile(r"/year=(\d{4})/month=(\d{2})/")

def get_year_month(path: str):
    m = ym.search(path)
    if not m:
        return None, None
    return int(m.group(1)), int(m.group(2))

# Read schemas (parquet footer only)
rows = []
for p in all_paths:
    y, m = get_year_month(p)
    sch = spark.read.parquet(p).schema
    for field in sch.fields:
        rows.append((y, m, p, field.name, field.dataType.simpleString(), field.nullable))

schema_inv = spark.createDataFrame(
    rows,
    schema=T.StructType([
        T.StructField("year", T.IntegerType(), True),
        T.StructField("month", T.IntegerType(), True),
        T.StructField("path", T.StringType(), False),
        T.StructField("col", T.StringType(), False),
        T.StructField("dtype", T.StringType(), False),
        T.StructField("nullable", T.BooleanType(), False),
    ])
)

schema_inv.cache()
schema_inv.count()
display(schema_inv.limit(50))

In [ ]:
# A signature of sorted column names per file/month
sig = (
    schema_inv.groupBy("year", "month", "path")
    .agg(F.sort_array(F.collect_set("col")).alias("cols"))
    .withColumn("schema_sig", F.sha2(F.concat_ws("||", F.col("cols")), 256))
)

# Count how many distinct schemas exist and which months use them
sig_summary = (
    sig.groupBy("schema_sig")
       .agg(
            F.count("*").alias("months"),
            F.min(F.struct("year","month")).alias("first"),
            F.max(F.struct("year","month")).alias("last")
        )
       .orderBy(F.desc("months"))
)

display(sig_summary)

# Inspect the columns for the top N schema variants
top_sigs = [r["schema_sig"] for r in sig_summary.limit(10).collect()]
for s in top_sigs:
    cols = sig.filter(F.col("schema_sig")==s).select("cols").limit(1).collect()[0]["cols"]
    print("\nSchema sig:", s, "columns:", len(cols))
    print(cols)


In [ ]:
# Which columns ever appear
all_cols = schema_inv.select("col").distinct().orderBy("col")
display(all_cols)

# Observed types per column
types_per_col = (
    schema_inv.groupBy("col")
              .agg(F.collect_set("dtype").alias("dtypes"), F.countDistinct("path").alias("files_with_col"))
              .orderBy(F.desc("files_with_col"))
)
display(types_per_col)


In [ ]:
import re
from pyspark.sql import functions as F, types as T

BASE = "Files/TLC_Trip_Record_Data"

# Hadoop FS recursive listing
jvm = spark._jvm
hconf = spark._jsc.hadoopConfiguration()
fs = jvm.org.apache.hadoop.fs.FileSystem.get(hconf)
Path = jvm.org.apache.hadoop.fs.Path

def list_parquet_files(root_path: str):
    out = []
    stack = [Path(root_path)]
    while stack:
        p = stack.pop()
        for status in fs.listStatus(p):
            if status.isDirectory():
                stack.append(status.getPath())
            else:
                s = status.getPath().toString()
                if s.lower().endswith(".parquet"):
                    out.append(s)
    return out

paths = list_parquet_files(BASE)
print("Files:", len(paths))

ym = re.compile(r"/year=(\d{4})/month=(\d{2})/")

def year_month_from_path(p):
    m = ym.search(p)
    return (int(m.group(1)), int(m.group(2))) if m else (None, None)

# Build signature per file based on columns only (fast)
rows = []
for p in paths:
    sch = spark.read.parquet(p).schema
    cols = sorted([f.name for f in sch.fields])
    sig = "|".join(cols)  # simple signature; stable enough
    y, m = year_month_from_path(p)
    rows.append((sig, y, m, p, len(cols)))

sig_df = spark.createDataFrame(rows, ["schema_sig", "year", "month", "path", "ncols"])
display(sig_df.groupBy("schema_sig").count().orderBy(F.desc("count")))

In [ ]:
from pyspark.sql import DataFrame

def standardize(df_in: DataFrame) -> DataFrame:
    # derive year/month from path for every slice
    df = (
        df_in.withColumn("_path", F.input_file_name())
             .withColumn("year", F.regexp_extract(F.col("_path"), r"/year=(\d{4})/", 1).cast("int"))
             .withColumn("month", F.regexp_extract(F.col("_path"), r"/month=(\d{2})/", 1).cast("int"))
             .drop("__index_level_0__")
    )

    cols = set(df.columns)

    def c(name: str):
        return F.col(name) if name in cols else F.lit(None)

    def co(*names):
        use = [F.col(n) for n in names if n in cols]
        return F.coalesce(*use) if use else F.lit(None)

    vendor_name_in = c("vendor_name").cast("string")
    vendorid_in = c("VendorID").cast("int")
    vendor_id_in = c("vendor_id").cast("int")

    vendorid = (
        F.when(vendorid_in.isNotNull(), vendorid_in)
         .when(vendor_id_in.isNotNull(), vendor_id_in)
         .when(vendor_name_in.isin("CMT"), F.lit(1))
         .when(vendor_name_in.isin("VTS"), F.lit(2))
         .otherwise(F.lit(None).cast("int"))
    )

    out = (
        df
        .withColumn("VendorID", vendorid)
        .withColumn("vendor_name_raw", vendor_name_in)

        # Modern
        .withColumn("tpep_pickup_datetime", c("tpep_pickup_datetime").cast("timestamp"))
        .withColumn("tpep_dropoff_datetime", c("tpep_dropoff_datetime").cast("timestamp"))
        .withColumn("PULocationID", c("PULocationID").cast("int"))
        .withColumn("DOLocationID", c("DOLocationID").cast("int"))
        .withColumn("RatecodeID", c("RatecodeID").cast("int"))
        .withColumn("store_and_fwd_flag", c("store_and_fwd_flag").cast("string"))
        .withColumn("payment_type", c("payment_type").cast("int"))
        .withColumn("passenger_count", c("passenger_count").cast("int"))
        .withColumn("trip_distance", c("trip_distance").cast("double"))

        .withColumn("fare_amount", c("fare_amount").cast("double"))
        .withColumn("extra", c("extra").cast("double"))
        .withColumn("mta_tax", c("mta_tax").cast("double"))
        .withColumn("tip_amount", c("tip_amount").cast("double"))
        .withColumn("tolls_amount", c("tolls_amount").cast("double"))
        .withColumn("improvement_surcharge", c("improvement_surcharge").cast("double"))
        .withColumn("total_amount", c("total_amount").cast("double"))
        .withColumn("congestion_surcharge", c("congestion_surcharge").cast("double"))
        .withColumn("airport_fee", co("airport_fee", "Airport_fee").cast("double"))
        .withColumn("cbd_congestion_fee", c("cbd_congestion_fee").cast("double"))

        # Legacy GPS + legacy datetime
        .withColumn("pickup_datetime_legacy", co("pickup_datetime", "Trip_Pickup_DateTime").cast("timestamp"))
        .withColumn("dropoff_datetime_legacy", co("dropoff_datetime", "Trip_Dropoff_DateTime").cast("timestamp"))
        .withColumn("pickup_longitude", co("pickup_longitude", "Start_Lon").cast("double"))
        .withColumn("pickup_latitude",  co("pickup_latitude",  "Start_Lat").cast("double"))
        .withColumn("dropoff_longitude",co("dropoff_longitude","End_Lon").cast("double"))
        .withColumn("dropoff_latitude", co("dropoff_latitude", "End_Lat").cast("double"))

        # Legacy numeric -> modern
        .withColumn("fare_amount", F.coalesce(F.col("fare_amount"), c("Fare_Amt").cast("double")))
        .withColumn("tip_amount", F.coalesce(F.col("tip_amount"), c("Tip_Amt").cast("double")))
        .withColumn("tolls_amount", F.coalesce(F.col("tolls_amount"), c("Tolls_Amt").cast("double")))
        .withColumn("total_amount", F.coalesce(F.col("total_amount"), c("Total_Amt").cast("double")))
        .withColumn("trip_distance", F.coalesce(F.col("trip_distance"), c("Trip_Distance").cast("double")))
        .withColumn("payment_type", F.coalesce(F.col("payment_type"), c("Payment_Type").cast("int")))
        .withColumn("passenger_count", F.coalesce(F.col("passenger_count"), c("Passenger_Count").cast("int")))
        .withColumn("RatecodeID", F.coalesce(F.col("RatecodeID"), co("rate_code", "Rate_Code").cast("int")))
        .withColumn("store_and_fwd_flag", F.coalesce(F.col("store_and_fwd_flag"), c("store_and_forward").cast("string")))
        .withColumn("extra", F.coalesce(F.col("extra"), c("surcharge").cast("double")))
    )

    # Introduced later fields
    out = (
        out
        .withColumn("improvement_surcharge",
                    F.when(F.col("year") < 2015, F.lit(None).cast("double")).otherwise(F.col("improvement_surcharge")))
        .withColumn("cbd_congestion_fee",
                    F.when(F.col("year") < 2025, F.lit(None).cast("double")).otherwise(F.col("cbd_congestion_fee")))
    )

    final_cols = [
        "VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime",
        "passenger_count", "trip_distance", "RatecodeID", "store_and_fwd_flag",
        "PULocationID", "DOLocationID", "payment_type",
        "fare_amount", "extra", "mta_tax", "tip_amount", "tolls_amount",
        "improvement_surcharge", "congestion_surcharge", "airport_fee", "cbd_congestion_fee",
        "total_amount",
        "vendor_name_raw",
        "pickup_datetime_legacy", "dropoff_datetime_legacy",
        "pickup_longitude", "pickup_latitude", "dropoff_longitude", "dropoff_latitude",
        "year", "month"
    ]
    return out.select(*final_cols)

In [ ]:
import re
from pyspark.sql import functions as F

BASE = "Files/TLC_Trip_Record_Data"

# Hadoop FS recursive listing
jvm = spark._jvm
hconf = spark._jsc.hadoopConfiguration()
fs = jvm.org.apache.hadoop.fs.FileSystem.get(hconf)
Path = jvm.org.apache.hadoop.fs.Path

def list_parquet_files(root_path: str):
    out = []
    stack = [Path(root_path)]
    while stack:
        p = stack.pop()
        for status in fs.listStatus(p):
            if status.isDirectory():
                stack.append(status.getPath())
            else:
                s = status.getPath().toString()
                if s.lower().endswith(".parquet"):
                    out.append(s)
    return out

paths = list_parquet_files(BASE)
print("Parquet files found:", len(paths))

ym = re.compile(r"/year=(\d{4})/month=(\d{2})/")
def ym_key(p):
    m = ym.search(p)
    return (int(m.group(1)), int(m.group(2))) if m else (9999, 99)

paths = sorted(paths, key=ym_key)

# Build a physical schema signature: "col:type|col:type|..."
def physical_sig(path: str) -> str:
    sch = spark.read.parquet(path).schema
    items = sorted([f"{f.name}:{f.dataType.simpleString()}" for f in sch.fields])
    return "|".join(items)

groups = {}
for p in paths:
    sig = physical_sig(p)
    groups.setdefault(sig, []).append(p)

# Overview
sizes = sorted([(k, len(v), v[0]) for k, v in groups.items()], key=lambda x: -x[1])
print("Distinct physical schema groups:", len(sizes))
print("Top groups (count, example):")
for _, n, ex in sizes[:10]:
    print(" ", n, ex)

In [ ]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

def standardize(df_in: DataFrame) -> DataFrame:
    df = (
        df_in.withColumn("_path", F.input_file_name())
             .withColumn("year", F.regexp_extract(F.col("_path"), r"/year=(\d{4})/", 1).cast("int"))
             .withColumn("month", F.regexp_extract(F.col("_path"), r"/month=(\d{2})/", 1).cast("int"))
             .drop("__index_level_0__")
    )

    cols = set(df.columns)

    def c(name: str):
        return F.col(name) if name in cols else F.lit(None)

    def co(*names):
        use = [F.col(n) for n in names if n in cols]
        return F.coalesce(*use) if use else F.lit(None)

    vendor_name_in = c("vendor_name").cast("string")
    vendor_id_in = c("vendor_id").cast("int")
    vendorid_in = c("VendorID").cast("int")

    vendorid = (
        F.when(vendorid_in.isNotNull(), vendorid_in)
         .when(vendor_id_in.isNotNull(), vendor_id_in)
         .when(vendor_name_in.isin("CMT"), F.lit(1))
         .when(vendor_name_in.isin("VTS"), F.lit(2))
         .otherwise(F.lit(None).cast("int"))
    )

    out = (
        df
        .withColumn("VendorID", vendorid)
        .withColumn("vendor_name_raw", vendor_name_in)

        # Modern columns
        .withColumn("tpep_pickup_datetime", c("tpep_pickup_datetime").cast("timestamp"))
        .withColumn("tpep_dropoff_datetime", c("tpep_dropoff_datetime").cast("timestamp"))
        .withColumn("PULocationID", c("PULocationID").cast("int"))
        .withColumn("DOLocationID", c("DOLocationID").cast("int"))
        .withColumn("RatecodeID", c("RatecodeID").cast("int"))
        .withColumn("store_and_fwd_flag", c("store_and_fwd_flag").cast("string"))
        .withColumn("payment_type", c("payment_type").cast("int"))
        .withColumn("passenger_count", c("passenger_count").cast("int"))
        .withColumn("trip_distance", c("trip_distance").cast("double"))

        .withColumn("fare_amount", c("fare_amount").cast("double"))
        .withColumn("extra", c("extra").cast("double"))
        .withColumn("mta_tax", c("mta_tax").cast("double"))
        .withColumn("tip_amount", c("tip_amount").cast("double"))
        .withColumn("tolls_amount", c("tolls_amount").cast("double"))
        .withColumn("improvement_surcharge", c("improvement_surcharge").cast("double"))
        .withColumn("total_amount", c("total_amount").cast("double"))
        .withColumn("congestion_surcharge", c("congestion_surcharge").cast("double"))
        .withColumn("airport_fee", co("airport_fee", "Airport_fee").cast("double"))
        .withColumn("cbd_congestion_fee", c("cbd_congestion_fee").cast("double"))

        # Legacy GPS + legacy datetime
        .withColumn("pickup_datetime_legacy", co("pickup_datetime", "Trip_Pickup_DateTime").cast("timestamp"))
        .withColumn("dropoff_datetime_legacy", co("dropoff_datetime", "Trip_Dropoff_DateTime").cast("timestamp"))
        .withColumn("pickup_longitude", co("pickup_longitude", "Start_Lon").cast("double"))
        .withColumn("pickup_latitude",  co("pickup_latitude",  "Start_Lat").cast("double"))
        .withColumn("dropoff_longitude",co("dropoff_longitude","End_Lon").cast("double"))
        .withColumn("dropoff_latitude", co("dropoff_latitude", "End_Lat").cast("double"))

        # Legacy numeric -> modern
        .withColumn("fare_amount", F.coalesce(F.col("fare_amount"), c("Fare_Amt").cast("double")))
        .withColumn("tip_amount", F.coalesce(F.col("tip_amount"), c("Tip_Amt").cast("double")))
        .withColumn("tolls_amount", F.coalesce(F.col("tolls_amount"), c("Tolls_Amt").cast("double")))
        .withColumn("total_amount", F.coalesce(F.col("total_amount"), c("Total_Amt").cast("double")))
        .withColumn("trip_distance", F.coalesce(F.col("trip_distance"), c("Trip_Distance").cast("double")))
        .withColumn("payment_type", F.coalesce(F.col("payment_type"), c("Payment_Type").cast("int")))
        .withColumn("passenger_count", F.coalesce(F.col("passenger_count"), c("Passenger_Count").cast("int")))
        .withColumn("RatecodeID", F.coalesce(F.col("RatecodeID"), co("rate_code", "Rate_Code").cast("int")))
        .withColumn("store_and_fwd_flag", F.coalesce(F.col("store_and_fwd_flag"), c("store_and_forward").cast("string")))
        .withColumn("extra", F.coalesce(F.col("extra"), c("surcharge").cast("double")))
    )

    # Introduced later fields
    out = (
        out
        .withColumn("improvement_surcharge",
                    F.when(F.col("year") < 2015, F.lit(None).cast("double")).otherwise(F.col("improvement_surcharge")))
        .withColumn("cbd_congestion_fee",
                    F.when(F.col("year") < 2025, F.lit(None).cast("double")).otherwise(F.col("cbd_congestion_fee")))
    )

    final_cols = [
        "VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime",
        "passenger_count", "trip_distance", "RatecodeID", "store_and_fwd_flag",
        "PULocationID", "DOLocationID", "payment_type",
        "fare_amount", "extra", "mta_tax", "tip_amount", "tolls_amount",
        "improvement_surcharge", "congestion_surcharge", "airport_fee", "cbd_congestion_fee",
        "total_amount",
        "vendor_name_raw",
        "pickup_datetime_legacy", "dropoff_datetime_legacy",
        "pickup_longitude", "pickup_latitude", "dropoff_longitude", "dropoff_latitude",
        "year", "month"
    ]
    return out.select(*final_cols)

In [ ]:
DELTA_TABLE = "tlc_yellow_tripdata_unified"

# Ensure Spark doesn't try to merge parquet schemas globally
spark.conf.set("spark.sql.parquet.mergeSchema", "false")

def write_batch(paths_batch, mode):
    df_batch = spark.read.format("parquet").load(paths_batch)
    df_std = standardize(df_batch)
    (df_std.write.format("delta")
          .mode(mode)
          .option("overwriteSchema", "true" if mode == "overwrite" else "false")
          .partitionBy("year", "month")
          .saveAsTable(DELTA_TABLE))

# Process groups largest-first (better progress)
group_items = sorted(groups.items(), key=lambda kv: -len(kv[1]))

written_groups = 0
failed_files = []

for gi, (sig, gpaths) in enumerate(group_items):
    mode = "overwrite" if gi == 0 else "append"
    try:
        print(f"Group {gi+1}/{len(group_items)}: files={len(gpaths)} mode={mode}")
        write_batch(gpaths, mode)
        written_groups += 1
    except Exception as e:
        print("Group failed, falling back to smaller batches:", repr(e))

        # Fallback: split into chunks of 10, then single file if needed
        chunk = 10
        for start in range(0, len(gpaths), chunk):
            sub = gpaths[start:start+chunk]
            try:
                write_batch(sub, "append" if gi > 0 or written_groups > 0 else "overwrite")
                written_groups += 1
            except Exception as e2:
                # Last resort: file-by-file for this subchunk only
                for p in sub:
                    try:
                        write_batch([p], "append" if gi > 0 or written_groups > 0 else "overwrite")
                        written_groups += 1
                    except Exception as e3:
                        failed_files.append((p, repr(e3)))
                        print("FAILED FILE:", p, repr(e3))

print("Done.")
print("Failed files:", len(failed_files))
if failed_files:
    for p, err in failed_files[:20]:
        print(" ", p, "->", err)

In [ ]:
# Size of source parquet files
BASE = "Files/TLC_Trip_Record_Data"

jvm = spark._jvm
hconf = spark._jsc.hadoopConfiguration()
fs = jvm.org.apache.hadoop.fs.FileSystem.get(hconf)
Path = jvm.org.apache.hadoop.fs.Path

def total_size_bytes(root_path: str):
    total = 0
    stack = [Path(root_path)]
    while stack:
        p = stack.pop()
        for status in fs.listStatus(p):
            if status.isDirectory():
                stack.append(status.getPath())
            else:
                total += status.getLen()
    return total

src_bytes = total_size_bytes(BASE)
src_gb = src_bytes / (1024**3)

print(f"Source Parquet size: {src_gb:.2f} GB")

In [ ]:
src_rows = spark.read.parquet("Files/TLC_Trip_Record_Data").count()
print(f"Source Parquet rows: {src_rows:,}")

In [ ]:
# Delta table storage path (Fabric managed table)
# DESCRIBE DETAIL gives the location
detail = spark.sql("DESCRIBE DETAIL tlc_yellow_tripdata_unified").collect()[0]
delta_path = detail["location"]

print("Delta location:", delta_path)

delta_bytes = total_size_bytes(delta_path)
delta_gb = delta_bytes / (1024**3)

print(f"Delta table size (data + log): {delta_gb:.2f} GB")

In [ ]:
delta_rows = spark.table("tlc_yellow_tripdata_unified").count()
print(f"Delta table rows: {delta_rows:,}")